<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# Advanced YouTube Downloader with Structured Storage
!pip install yt-dlp
!sudo apt install ffmpeg -qq

from google.colab import drive
import os
import yt_dlp

# ========== CONFIGURATION ========== #
class AdvancedConfig:
    DRIVE_PATH = "/content/drive/MyDrive/YouTube_Downloads"
    SUBTITLES = ['en', 'fa']  # English and Persian subtitles
    VIDEO_FORMAT = 'mp4'

# ========== CORE ENGINE ========== #
class YouTubeDownloader:
    def __init__(self, config):
        self.config = config
        self._setup_infrastructure()

    def _setup_infrastructure(self):
        drive.mount('/content/drive')
        os.makedirs(self.config.DRIVE_PATH, exist_ok=True)

    def download(self, url):
        ydl_opts = self._build_options()
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

    def _build_options(self):
        return {
            # === FORMAT SELECTION === #
            'format': f'bestvideo[ext={self.config.VIDEO_FORMAT}]+bestaudio/best',
            'merge_output_format': self.config.VIDEO_FORMAT,

            # === SUBTITLE CONFIG === #
            'writesubtitles': True,
            'subtitleslangs': self.config.SUBTITLES,
            'writeautomaticsub': True,
            'embedsubtitles': True,

            # === FOLDER STRUCTURE === #
            'outtmpl': {
                'default': f'{self.config.DRIVE_PATH}/%(playlist_title)s/%(playlist_index)s - %(title)s/%(title)s.%(ext)s',
                'pl_thumbnail': f'{self.config.DRIVE_PATH}/%(playlist_title)s/playlist_thumbnail.%(ext)s',
                'thumbnail': f'{self.config.DRIVE_PATH}/%(playlist_title)s/%(playlist_index)s - %(title)s/thumbnail.%(ext)s',
                'subtitle': f'{self.config.DRIVE_PATH}/%(playlist_title)s/%(playlist_index)s - %(title)s/%(language)s.%(ext)s',
            },

            # === POST-PROCESSING === #
            'postprocessors': [
                {'key': 'FFmpegVideoConvertor', 'preferedformat': 'mp4'},
                {'key': 'FFmpegEmbedSubtitle'},
                {'key': 'FFmpegMetadata'},
                {'key': 'EmbedThumbnail'},
                {
                    'key': 'Exec',
                    'exec_cmd': (
                        'mkdir -p "{dirname}" && '
                        'mv "{filepath}" "{dirname}" && '
                        'mv "{thumbpath}" "{dirname}" && '
                        'for sub in *.vtt; do mv "$sub" "{dirname}/${sub}"; done'
                    ),
                    'when': 'after_move'
                }
            ],

            # === METADATA === #
            'writethumbnail': True,
            'writeinfojson': True,

            # === ERROR HANDLING === #
            'ignoreerrors': True,
            'retries': 10,
            'download_archive': f'{self.config.DRIVE_PATH}/download_archive.txt',
        }

# ========== EXECUTION ========== #
if __name__ == "__main__":
    config = AdvancedConfig()
    engine = YouTubeDownloader(config)



In [ ]:
    # Example usage:
    content_urls = [
        "https://www.youtube.com/watch?v=kaZOXRqFPCw",  # Playlist URL         # Single video URL
    ]

    for url in content_urls:
        engine.download(url)

print("✅ Download completed with organized folder structure!")